# 🔗 Network Flow — Capacity and Routing Decisions (LP/MIP)

This notebook models how flow should be routed through a constrained network to satisfy demand at minimum total cost.

The emphasis is on **system-level decision quality** — understanding bottlenecks, binding constraints, and the value of incremental capacity — rather than on algorithmic details.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Solver: PuLP (open-source; runs without a commercial license)
try:
    import pulp
except ImportError as e:
    raise ImportError(
        "PuLP is required to run this notebook. Install with: pip install pulp"
    ) from e


## Modeling Assumptions

- **Flow conservation:** what enters an intermediate node must leave it (unless the node has supply or demand).
- **Supply and demand:** supply nodes inject flow; demand nodes require flow.
- **Capacity limits:** each route has an upper bound on how much flow it can carry.
- **Continuous flow:** flows can be fractional (LP). An extension introduces discrete capacity choices (MIP).

These assumptions reflect a broad class of real systems (logistics, supply chains, infrastructure networks).


In [ ]:
# ---- Network definition (manager-readable names) ----

nodes = ['Supplier A', 'Supplier B', 'Hub C', 'Customer D', 'Customer E']

# Supply (+) and demand (-) at each node (units per period)
supply = pd.Series([17.0, 13.0, 0.0, 0.0, 0.0], index=nodes, name="supply")
demand = pd.Series([0.0, 0.0, 0.0, 22.0, 8.0], index=nodes, name="demand")

# Existing directed routes (arcs)
arcs = [('Supplier A', 'Supplier B'), ('Supplier A', 'Hub C'), ('Supplier A', 'Customer D'), ('Supplier B', 'Hub C'), ('Supplier B', 'Customer E'), ('Hub C', 'Customer D'), ('Hub C', 'Customer E'), ('Customer E', 'Customer D')]

# Unit transportation / handling cost on each arc
arc_unit_cost = {
    ('Supplier A', 'Supplier B'): 12.0,
    ('Supplier A', 'Hub C'): 14.0,
    ('Supplier A', 'Customer D'): 16.0,
    ('Supplier B', 'Hub C'): 8.0,
    ('Supplier B', 'Customer E'): 13.0,
    ('Hub C', 'Customer D'): 4.0,
    ('Hub C', 'Customer E'): 14.0,
    ('Customer E', 'Customer D'): 12.0,
}

LOW_CAPACITY = 9   # baseline capacity per arc
HIGH_CAPACITY = 18 # expanded capacity per arc (used in the MIP extension)

LOW_CAP_FIXED_COST  = 100   # fixed cost to open arc at LOW capacity
HIGH_CAP_FIXED_COST = 150  # fixed cost to open arc at HIGH capacity

# Sanity check: total supply must match total demand
total_supply = supply.sum()
total_demand = demand.sum()

display(pd.DataFrame({"supply": supply, "demand": demand}))
print(f"Total supply: {total_supply:.0f} | Total demand: {total_demand:.0f}")
assert abs(total_supply - total_demand) < 1e-9, "Total supply must equal total demand for feasibility."


## Model 1 — Baseline LP (fixed arc capacities)

**Decision:** how much flow to send on each available route.

**Objective:** minimize total cost while satisfying all demands and respecting route capacities.

This answers: *Given the network as-is, what is the lowest-cost feasible routing plan?*


In [ ]:
# ---- Model 1: Linear Program (LP) ----

lp = pulp.LpProblem("NetworkFlow_BaselineLP", pulp.LpMinimize)

# Decision variables: flow on each arc (bounded by baseline capacity)
flow = pulp.LpVariable.dicts(
    "flow",
    arcs,
    lowBound=0,
    upBound=LOW_CAPACITY,
    cat="Continuous",
)

# Objective: minimize total unit cost
lp += pulp.lpSum(arc_unit_cost[a] * flow[a] for a in arcs)

# Flow balance at each node:
# inflow + supply = outflow + demand
for node in nodes:
    inflow  = pulp.lpSum(flow[(i, node)] for (i, j) in arcs if j == node)
    outflow = pulp.lpSum(flow[(node, j)] for (i, j) in arcs if i == node)
    lp += inflow + supply[node] == outflow + demand[node], f"flow_balance_{node}"

# Solve
lp.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[lp.status])
print("Total cost:", pulp.value(lp.objective))


In [ ]:
# ---- Results: Baseline LP ----

flow_values = pd.DataFrame(
    [{"from": i, "to": j, "flow": flow[(i, j)].value(), "unit_cost": arc_unit_cost[(i, j)]} for (i, j) in arcs]
)

flow_values["capacity"] = LOW_CAPACITY
flow_values["utilization"] = flow_values["flow"] / flow_values["capacity"]
flow_values["arc_cost"] = flow_values["flow"] * flow_values["unit_cost"]

flow_values = flow_values.sort_values(["utilization", "arc_cost"], ascending=False)

display(flow_values.reset_index(drop=True))

# Identify likely bottlenecks (near-full arcs)
bottlenecks = flow_values[flow_values["utilization"] >= 0.99][["from", "to", "flow", "capacity", "unit_cost"]]
print("\nBottleneck arcs (utilization ≥ 99%):")
display(bottlenecks.reset_index(drop=True))

# Simple visualization: utilization by arc
plt.figure()
plt.bar(range(len(flow_values)), flow_values["utilization"])
plt.xticks(range(len(flow_values)), [f"{r['from']}→{r['to']}" for _, r in flow_values.iterrows()], rotation=45, ha="right")
plt.ylabel("Utilization (flow / capacity)")
plt.title("Baseline LP — Arc Utilization")
plt.tight_layout()
plt.show()


## Model 2 — Capacity Choice (MIP extension)

**Decision:** for each route, choose **LOW** or **HIGH** capacity (or keep it closed), paying a fixed cost for the capacity level selected.

This answers: *Where is incremental capacity worth paying for — and how does the optimal routing change?*


In [ ]:
# ---- Model 2: Mixed-Integer Program (MIP) ----

mip = pulp.LpProblem("NetworkFlow_CapacityChoiceMIP", pulp.LpMinimize)

# Flow on arcs (continuous)
flow2 = pulp.LpVariable.dicts("flow", arcs, lowBound=0, cat="Continuous")

# Capacity selection (binary)
use_low  = pulp.LpVariable.dicts("use_low", arcs, lowBound=0, upBound=1, cat="Binary")
use_high = pulp.LpVariable.dicts("use_high", arcs, lowBound=0, upBound=1, cat="Binary")

# Objective = variable cost + fixed capacity costs
mip += (
    pulp.lpSum(arc_unit_cost[a] * flow2[a] for a in arcs)
    + pulp.lpSum(LOW_CAP_FIXED_COST  * use_low[a]  for a in arcs)
    + pulp.lpSum(HIGH_CAP_FIXED_COST * use_high[a] for a in arcs)
)

# At most one capacity level per arc (or closed)
for a in arcs:
    mip += use_low[a] + use_high[a] <= 1, f"cap_choice_{a[0]}_{a[1]}"

# Capacity constraint ties flow to chosen level
for a in arcs:
    mip += flow2[a] <= LOW_CAPACITY  * use_low[a] + HIGH_CAPACITY * use_high[a], f"cap_limit_{a[0]}_{a[1]}"

# Flow balance
for node in nodes:
    inflow  = pulp.lpSum(flow2[(i, node)] for (i, j) in arcs if j == node)
    outflow = pulp.lpSum(flow2[(node, j)] for (i, j) in arcs if i == node)
    mip += inflow + supply[node] == outflow + demand[node], f"flow_balance_{node}"

# Solve
mip.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[mip.status])
print("Total cost (incl. fixed capacity costs):", pulp.value(mip.objective))


In [ ]:
# ---- Results: Capacity Choice MIP ----

rows = []
for (i, j) in arcs:
    low  = use_low[(i, j)].value()
    high = use_high[(i, j)].value()
    cap = (LOW_CAPACITY * low) + (HIGH_CAPACITY * high)
    rows.append({
        "from": i,
        "to": j,
        "flow": flow2[(i, j)].value(),
        "unit_cost": arc_unit_cost[(i, j)],
        "cap_selected": "HIGH" if high > 0.5 else ("LOW" if low > 0.5 else "CLOSED"),
        "capacity": cap,
        "utilization": (flow2[(i, j)].value() / cap) if cap > 0 else np.nan,
        "fixed_cost": (LOW_CAP_FIXED_COST * low) + (HIGH_CAP_FIXED_COST * high),
        "variable_cost": arc_unit_cost[(i, j)] * flow2[(i, j)].value(),
    })

mip_df = pd.DataFrame(rows).sort_values(["cap_selected", "utilization"], ascending=[True, False])
display(mip_df.reset_index(drop=True))

chosen = mip_df[mip_df["cap_selected"] != "CLOSED"][["from","to","cap_selected","capacity","flow","utilization","fixed_cost"]]
print("\nSelected arcs and capacity levels:")
display(chosen.reset_index(drop=True))

# Compare objective components
total_fixed = mip_df["fixed_cost"].sum()
total_variable = mip_df["variable_cost"].sum()
print(f"\nCost breakdown → variable: {total_variable:,.2f} | fixed: {total_fixed:,.2f} | total: {total_variable+total_fixed:,.2f}")


## Managerial Interpretation

- **Bottlenecks dominate:** A small number of near-capacity arcs determine feasibility and cost.
- **System effects matter:** Local “cheap” routes may be unusable if upstream capacity binds.
- **Targeted capacity wins:** Paying to expand non-binding arcs provides little value; expanding bottlenecks can change the optimal routing plan.

This is the core value of network flow models: making system constraints and tradeoffs explicit and defensible.
